# 03 — Virtual Encoder (video -> trajectory)

Convert a recording of the machine trajectory into position coordinates (mm) per frame.

**Input:** video `pose_video_XX.avi` from `data/raw/video_*` (configured in `config_paths.py`)
**Other input:** `data/processed/camera_calibration/*.csv` (K and D from stage 01)

**Output:** `data/processed/virtual_encoder/<CSV_PREFIX>_XX.csv` with columns
`Frame, Waktu (detik), X Posisi (mm), Y Posisi (mm), Reprojection Error (px)`.

The last cell produces a baseline vs calibrated comparison figure for the manuscript
(output to `figures/`).

In [ ]:
# ==========================================================================
# LOAD PATH CONFIGURATION  (single source for all notebooks: config_paths.py)
# ==========================================================================
import os
import sys

_p = os.path.abspath(os.getcwd())
while not os.path.isfile(os.path.join(_p, "config_paths.py")):
    _parent = os.path.dirname(_p)
    if _parent == _p:
        raise FileNotFoundError(
            "config_paths.py not found. Run this notebook from inside the "
            "curated repo folder, or set the KALIBRASI_REPO environment variable to the repo root."
        )
    _p = _parent
sys.path.insert(0, _p)
import config_paths as CFG

CFG.diagnose()
print()


In [ ]:
# ==========================================================================
# LOAD CAMERA PARAMETERS (K and D) FROM THE CAMERA CALIBRATION CSV OUTPUT
# ==========================================================================
import numpy as np
import pandas as pd

CAM_DIR = os.path.join(CFG.OUTPUT_DIR, "camera_calibration")
_HINT = ("First run 01_camera_calibration/camera_calibration.ipynb until "
         "the export cell finishes.")
for _f in ("camera_matrix.csv", "dist_coeffs.csv", "camera_params.csv"):
    if not os.path.isfile(os.path.join(CAM_DIR, _f)):
        raise FileNotFoundError(f"Not found: {os.path.join(CAM_DIR, _f)}\n{_HINT}")

camera_matrix = np.loadtxt(os.path.join(CAM_DIR, "camera_matrix.csv"), delimiter=",", ndmin=2)
dist_coeffs = np.loadtxt(os.path.join(CAM_DIR, "dist_coeffs.csv"), delimiter=",", ndmin=2).reshape(1, -1)
_p = pd.read_csv(os.path.join(CAM_DIR, "camera_params.csv")).set_index("parameter")["value"]
img_shape = (int(_p["image_width"]), int(_p["image_height"]))
K, D = camera_matrix, dist_coeffs
print(f"K and D loaded from {CAM_DIR}  | image_size={img_shape}")
print("K =\n", np.round(K, 4))


In [ ]:
# 7. VIRTUAL ENCODER (EDLINES + HOMOGRAPHY ALIGNMENT & EXPORT CSV) BATCH PROCESSING
# ==========================================
import cv2
import numpy as np
import matplotlib.pyplot as plt
import time
import pandas as pd

# --- 1. LOAD CAMERA PARAMETERS ---
try:
    K = camera_matrix 
    D = dist_coeffs
    print("✅ Camera matrix and distortion loaded successfully.")
except NameError:
    raise NameError("ERROR: Run the Cell containing the camera calibration (Cell 7) first!")

CHESSBOARD_SIZE = (7, 7)
SQUARE_SIZE = 20.0 # mm
expected_pts = CHESSBOARD_SIZE[0] * CHESSBOARD_SIZE[1]

# Prepare object points (0,0,0), (20,0,0), ...
objp = np.zeros((expected_pts, 3), np.float32)
objp[:, :2] = np.mgrid[0:CHESSBOARD_SIZE[0], 0:CHESSBOARD_SIZE[1]].T.reshape(-1, 2) * SQUARE_SIZE

# --- 2. BATCH PROCESSING LOOP ---
# List of file indices to test
FILE_INDICES = CFG.FILE_INDICES
# FILE_INDICES = [1,2,3]  # old version; now from config_paths.py
# Loop over the given list of indices
for video_index in FILE_INDICES:
    
    # Dynamic path setup using 02d (so 1 becomes 01, 15 stays 15)
    video_filename = f"pose_video_{video_index:02d}.avi"
    VIDEO_PATH = os.path.join(CFG.VIDEO_DIR, CFG.VIDEO_PATTERN.format(video_index))
    
    # Set up output file names
    CSV_OUTPUT_PATH = os.path.join(CFG.OUTPUT_DIR, "virtual_encoder",
                                   f"{CFG.CSV_PREFIX}_{video_index:02d}.csv")
    PNG_OUTPUT_PATH = os.path.join(CFG.FIG_DIR,
                                   f"{CFG.CSV_PREFIX}_{video_index:02d}.png")
    
    print(f"\n{'='*60}")
    print(f"🔄 STARTING VIDEO PROCESSING: {video_filename}")
    print(f"{'='*60}")

    cap = cv2.VideoCapture(VIDEO_PATH)

    if not cap.isOpened():
        print(f"❌ Error: Video cannot be opened at path:\n{VIDEO_PATH}")
        continue # Move on to the next video in the list if it fails to open
    
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"Processing {total_frames} frames @ {fps} FPS")
    print("-" * 50)

    # Containers (Reset for each video)
    x_traj, y_traj, timestamps = [], [], []
    rmse_list = []
    export_data = []
    
    # Counters for statistics (Reset for each video)
    frame_count = 0
    valid_frames = 0
    skipped_rmse = 0      # Frame detected but RMSE > threshold
    failed_detection = 0  # Frame that failed EDLines/Sorting detection
    
    tvec_0 = None 
    threshold_rmse = 0.3 # Accuracy threshold in pixels
    start_time = time.time()

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
            
        frame_count += 1
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        
        # --- STEP A: EDLINES & CORNERS DETECTION ---
        lines = detect_edlines_raw(gray)
        initial_corners, _ = get_initial_corners_strict_paper(gray, lines)
        
        # Filter Logic 1: Check whether the number of points is sufficient
        if initial_corners is not None and len(initial_corners) >= expected_pts:
            refined_corners = optimize_subpixel_gradient(gray, initial_corners)
            sorted_corners = sort_corners_robust_paper(refined_corners, gray, CHESSBOARD_SIZE)
            
            # Filter Logic 2: Check whether sorting succeeded
            if sorted_corners is not None:
                corners_final = sorted_corners.reshape(-1, 1, 2)
                
                # --- STEP B: POSE ESTIMATION ---
                success, rvec, tvec = cv2.solvePnP(objp, corners_final, K, D)
                
                if success:
                    # Compute Reprojection Error
                    imgpts2, _ = cv2.projectPoints(objp, rvec, tvec, K, D)
                    norm_l2 = cv2.norm(corners_final, imgpts2, cv2.NORM_L2)
                    rmse = np.sqrt((norm_l2 ** 2) / len(imgpts2))
                    
                    # Filter Logic 3: Check RMSE Threshold
                    if rmse <= threshold_rmse:
                        if tvec_0 is None:
                            tvec_0 = tvec.copy()
                            print(f"-> Origin (0,0) set at frame {frame_count}")
                        
                        # Compute relative translation (mm)
                        x_rel = tvec[0][0] - tvec_0[0][0]
                        y_rel = tvec[1][0] - tvec_0[1][0]
                        current_time_sec = frame_count / fps
                        
                        # Store data
                        x_traj.append(x_rel)
                        y_traj.append(y_rel)
                        timestamps.append(current_time_sec)
                        rmse_list.append(rmse)
                        
                        export_data.append({
                            "Frame": frame_count,
                            "Waktu (detik)": round(current_time_sec, 4),
                            "X Posisi (mm)": round(x_rel, 4),
                            "Y Posisi (mm)": round(y_rel, 4),
                            "Reprojection Error (px)": round(rmse, 4)
                        })
                        valid_frames += 1
                    else:
                        skipped_rmse += 1
                else:
                    failed_detection += 1
            else:
                failed_detection += 1 # Failed in sorting
        else:
            failed_detection += 1 # Failed in initial detection (EDLines/Number of points)
        
        # Progress Monitor
        if frame_count % 50 == 0:
            elapsed = time.time() - start_time
            print(f"Frame {frame_count}/{total_frames} | Valid: {valid_frames} | Failed: {failed_detection}")

    cap.release()
    print("-" * 50)
    
    # --- 3. EXPORT & SUMMARY FOR THE CURRENT VIDEO ---
    if valid_frames > 0:
        avg_rmse = np.mean(rmse_list)
        df = pd.DataFrame(export_data)
        df.to_csv(CSV_OUTPUT_PATH, index=False)
        
        print(f"FINAL PROCESSING RESULT ({video_filename}):")
        print(f"1. Total Video Frames      : {frame_count}")
        print(f"2. Successful Frames (Valid) : {valid_frames}")
        print(f"3. Skipped (RMSE > {threshold_rmse}) : {skipped_rmse}")
        print(f"4. Failed Feature Detection    : {failed_detection}")
        print(f"--------------------------------------")
        print(f"Total Check (2+3+4)       : {valid_frames + skipped_rmse + failed_detection}")
        print(f"MEAN RMSE            : {avg_rmse:.4f} px")
        print(f"✅ CSV file saved      : {CSV_OUTPUT_PATH}")
        
        # --- 4. VISUALIZATION ---
        plt.figure(figsize=(10, 7))
        sc = plt.scatter(x_traj, y_traj, c=timestamps, cmap='viridis', s=15, label='Trajectory')
        plt.colorbar(sc, label='Time (Seconds)')
        plt.plot(x_traj, y_traj, color='black', alpha=0.2)
        
        plt.plot(x_traj[0], y_traj[0], 'go', markersize=10, label='Start')
        plt.plot(x_traj[-1], y_traj[-1], 'ro', markersize=10, label='End')
        
        plt.title(f'2D Trajectory - {video_filename}', fontweight='bold')
        plt.xlabel('X (mm)')
        plt.ylabel('Y (mm)')
        plt.axis('equal')
        plt.grid(True, linestyle=':', alpha=0.6)
        plt.legend()
        
        # Save the plot as PNG (Must be before plt.show)
        plt.savefig(PNG_OUTPUT_PATH, dpi=300, bbox_inches='tight')
        print(f"✅ Plot saved          : {PNG_OUTPUT_PATH}")
        
        plt.show() 
    else:
        print(f"❌ No frames were successfully processed for {video_filename}.")

print("\n🎉 ALL VIDEOS FROM THE LIST HAVE BEEN PROCESSED AND SAVED.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

# List of file indices processed earlier
FILE_INDICES = [1]

print("🔄 STARTING RE-PLOT PROCESS FROM CSV...")

for video_index in FILE_INDICES:
    # Match the CSV file name to the format saved earlier
    csv_filename = os.path.join(CFG.CSV_VE_DIR, "baseline_x_analysis.csv")
    
    # Check whether the CSV file actually exists in the folder
    if not os.path.exists(csv_filename):
        print(f"\n❌ File {csv_filename} not found. Skipping...")
        continue
        
    # 1. READ DATA FROM CSV
    df = pd.read_csv(csv_filename)
    
    # 2. COMPUTE STATISTICS FROM THE DATAFRAME
    # Count the number of successfully processed frames (number of rows in the CSV)
    valid_frames = len(df)
    
    # Compute the mean of the Reprojection Error column
    avg_rmse = df["Reprojection Error (px)"].mean()
    
    print(f"\n{'='*50}")
    print(f"📊 STATISTICS FROM: {csv_filename}")
    print(f"Total Valid Frames : {valid_frames}")
    print(f"Mean RMSE    : {avg_rmse:.4f} px")
    print(f"{'='*50}")
    
    # 3. CREATE RE-PLOT
    plt.figure(figsize=(10, 7))
    
    # Take the data columns for the X, Y, and Time axes
    x_traj = df["X Posisi (mm)"]
    y_traj = df["Y Posisi (mm)"]
    timestamps = df["Waktu (detik)"]
    
    sc = plt.scatter(x_traj, y_traj, c=timestamps, cmap='viridis', s=15, label='Trajectory')
    plt.colorbar(sc, label='Time (Seconds)')
    
    # Thin connecting line between points
    plt.plot(x_traj, y_traj, color='black', alpha=0.2)
    
    # Mark the start point (index 0) and end point (last index / -1)
    # Use .iloc to take values by row position
    plt.plot(x_traj.iloc[0], y_traj.iloc[0], 'go', markersize=10, label='Start')
    plt.plot(x_traj.iloc[-1], y_traj.iloc[-1], 'ro', markersize=10, label='End')
    
    plt.title(f'2D Trajectory (Re-plot) - Index {video_index:02d}', fontweight='bold')
    plt.xlabel('X (mm)')
    plt.ylabel('Y (mm)')
    plt.axis('equal')
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.legend()
    
    # 4. SAVE THE NEW PLOT RESULT
    # File name slightly differs so it does not overwrite the original plot (optional)
    png_filename = os.path.join(CFG.FIG_DIR, f"baseline_plot{video_index:02d}.png")
    plt.savefig(png_filename, dpi=300, bbox_inches='tight')
    print(f"✅ Re-plot saved: {png_filename}")
    
    # Close the figure from memory so computation stays light
    # Change to plt.show() if you want to see them pop up one by one
    plt.close()

print("\n🎉 ALL CSV FILES HAVE BEEN RE-PROCESSED.")

In [ ]:
#THIS IS THE CLOSEST TO 10
import os
import matplotlib.pyplot as plt
import pandas as pd

# 1. PUT 2 DIFFERENT CSV FILE PATHS & NAMES HERE
csv_file1 = CFG.CSV_NASKAH["baseline_y"]
csv_file2 = CFG.CSV_NASKAH["calibrated_x_analysis"]
files_to_process = [
    {"path": csv_file1, "label": "Baseline"},
    {"path": csv_file2, "label": "Calibration Result"},
]

print("🔄 STARTING COMBINED PLOT PROCESS (TARGET NEAR ±10 mm)...")

# Canvas is made wide (horizontal length 10, vertical height 5)
plt.figure(figsize=(10, 5))

# Horizontal Ground Truth line at Y=0 (because the original X axis is now on the vertical Y axis)
plt.axhline(y=0, color='black', linewidth=2, linestyle='--', label='Ground Truth')

colors = ['blue', 'red']

for i, item in enumerate(files_to_process):
    csv_filename = item["path"]
    label_name = item["label"]
    
    if not os.path.exists(csv_filename):
        continue
        
    df = pd.read_csv(csv_filename)
    
    # Take the original data
    x_asli = df["X Posisi (mm)"]
    y_asli = df["Y Posisi (mm)"]
    
    # PLOT TRANSPOSED TRAJECTORY (original Y becomes the plot X axis, original X becomes the plot Y axis)
    plt.plot(
        y_asli, x_asli, 
        color=colors[i % len(colors)], 
        linewidth=2, label=label_name, zorder=2
    )
    
    warna_plot = colors[i % len(colors)]
    
    # =========================================================================
    # 1. POINT CLOSEST TO TARGET -10 mm (LEFT)
    # =========================================================================
    # Find the index with the smallest absolute difference from -10
    idx_target_neg = (df["Y Posisi (mm)"] - (-10)).abs().idxmin()
    x_kiri = df.loc[idx_target_neg, "Y Posisi (mm)"]  # X axis in the plot
    y_kiri = df.loc[idx_target_neg, "X Posisi (mm)"]  # Y axis in the plot
    
    plt.scatter(x_kiri, y_kiri, color=warna_plot, s=50, marker='o', edgecolor='black', zorder=5)
    
    # Baseline (Blue) text on top, Calibrated (Red) text below
    if i == 0:
        offset_kiri = (10, 25)
        va_align = 'bottom'
    else:
        offset_kiri = (10, -25)
        va_align = 'top'
        
    # Annotation still displays the original format (X_asli, Y_asli)
    plt.annotate(
        f"({y_kiri:.2f}, {x_kiri:.2f})", 
        (x_kiri, y_kiri),
        textcoords="offset points", 
        xytext=offset_kiri, 
        ha='left', va=va_align,
        fontsize=9, color=warna_plot, fontweight='bold',
        bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.8, ec="none"),
        arrowprops=dict(arrowstyle="-", color=warna_plot, lw=0.8)
    )
        
    # =========================================================================
    # 2. POINT CLOSEST TO TARGET +10 mm (RIGHT)
    # =========================================================================
    # Find the index with the smallest absolute difference from +10
    idx_target_pos = (df["Y Posisi (mm)"] - 10).abs().idxmin()
    x_kanan = df.loc[idx_target_pos, "Y Posisi (mm)"]  # X axis in the plot
    y_kanan = df.loc[idx_target_pos, "X Posisi (mm)"]  # Y axis in the plot
    
    plt.scatter(x_kanan, y_kanan, color=warna_plot, s=50, marker='o', edgecolor='black', zorder=5)
    
    # Baseline (Blue) text on top, Calibrated (Red) text below
    if i == 0:
        offset_kanan = (-10, 25)
        va_align_kanan = 'bottom'
    else:
        offset_kanan = (-10, -25)
        va_align_kanan = 'top'
        
    plt.annotate(
        f"({y_kanan:.2f}, {x_kanan:.2f})", 
        (x_kanan, y_kanan),
        textcoords="offset points", 
        xytext=offset_kanan, 
        ha='right', va=va_align_kanan,
        fontsize=9, color=warna_plot, fontweight='bold',
        bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.8, ec="none"),
        arrowprops=dict(arrowstyle="-", color=warna_plot, lw=0.8)
    )

plt.title('2D Combined Trajectory (Nearest Target ±10 mm)', fontweight='bold')

# Axis labels
plt.xlabel('Original Y Position (mm)')
plt.ylabel('Original X Position (mm)')

# Axis limits so the visualization stays proportional and tidy
plt.ylim(-1, 1)
plt.xlim(-12, 12)

plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc='lower center')

# SAVE THE PLOT RESULT
png_filename = os.path.join(CFG.FIG_DIR, "PLOT_GABUNGAN_TRANSPOSE.png")
plt.savefig(png_filename, dpi=300, bbox_inches='tight')
print(f"\n✅ Horizontal plot saved successfully: {png_filename}")
plt.close()

In [ ]:
import os
import matplotlib.pyplot as plt
import pandas as pd

# 1. PUT 2 DIFFERENT CSV FILE PATHS & NAMES HERE
csv_file1 = CFG.CSV_NASKAH["baseline_y"]
csv_file2 = CFG.CSV_NASKAH["calibrated_y"]

files_to_process = [
    {"path": csv_file1, "label": "Baseline"},
    {"path": csv_file2, "label": "Calibration Result"},
]

print("🔄 STARTING COMBINED PLOT PROCESS (1:1 SQUARE)...")

# Canvas is made square (10x10)
plt.figure(figsize=(10, 10))

# Vertical Ground Truth line at X=0
plt.axvline(x=0, color='black', linewidth=2, linestyle='--', label='Ground Truth')

colors = ['blue', 'red']

for i, item in enumerate(files_to_process):
    csv_filename = item["path"]
    label_name = item["label"]
    
    if not os.path.exists(csv_filename):
        print(f"⚠️ File not found: {csv_filename}")
        continue
        
    df = pd.read_csv(csv_filename)
    
    # Take the original data
    x_asli = df["X Posisi (mm)"]
    y_asli = df["Y Posisi (mm)"]
    
    # PLOT NATURAL TRAJECTORY
    plt.plot(
        x_asli, y_asli, 
        color=colors[i % len(colors)], 
        linewidth=1.0, label=label_name, zorder=2
    )
    
    warna_plot = colors[i % len(colors)]
    
    # =========================================================================
    # 1. BOTTOM END POINT (MINIMUM Y_ASLI)
    # =========================================================================
    idx_min_y = df["Y Posisi (mm)"].idxmin()
    x_bawah = df.loc[idx_min_y, "X Posisi (mm)"]  
    y_bawah = df.loc[idx_min_y, "Y Posisi (mm)"]  
    
    plt.scatter(x_bawah, y_bawah, color=warna_plot, s=50, marker='o', edgecolor='black', zorder=5)
    
    # Baseline (Blue) text on the left, Calibrated (Red) text on the right (spreading outward)
    if i == 0:
        offset_bawah = (-30, -20)
        ha_align_bawah = 'right'
    else:
        offset_bawah = (30, -20)
        ha_align_bawah = 'left'
        
    plt.annotate(
        f"({x_bawah:.2f}, {y_bawah:.2f})", 
        (x_bawah, y_bawah),
        textcoords="offset points", 
        xytext=offset_bawah, 
        ha=ha_align_bawah, va='top',
        fontsize=12, color=warna_plot, fontweight='bold',
        bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.8, ec="none"),
        arrowprops=dict(arrowstyle="-", color=warna_plot, lw=0.5)
    )
        
    # =========================================================================
    # 2. TOP END POINT (MAXIMUM Y_ASLI)
    # =========================================================================
    idx_max_y = df["Y Posisi (mm)"].idxmax()
    x_atas = df.loc[idx_max_y, "X Posisi (mm)"]  
    y_atas = df.loc[idx_max_y, "Y Posisi (mm)"]  
    
    plt.scatter(x_atas, y_atas, color=warna_plot, s=50, marker='o', edgecolor='black', zorder=5)
    
    # ✅ FIX HERE:
    # Baseline (Blue) now on the right, Calibrated (Red) on the left
    if i == 0:
        offset_atas = (30, 20)   # Positive X offset (Push to the Right)
        ha_align_atas = 'left'
    else:
        offset_atas = (-30, 20)  # Negative X offset (Push to the Left)
        ha_align_atas = 'right'
        
    plt.annotate(
        f"({x_atas:.2f}, {y_atas:.2f})", 
        (x_atas, y_atas),
        textcoords="offset points", 
        xytext=offset_atas, 
        ha=ha_align_atas, va='bottom',
        fontsize=12, color=warna_plot, fontweight='bold',
        bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.8, ec="none"),
        arrowprops=dict(arrowstyle="-", color=warna_plot, lw=0.5)
    )

plt.title('2D Trajectory Y Axis', fontweight='bold')
plt.xlabel('X Posisi (mm)')
plt.ylabel('Y Posisi (mm)')

# Axis limits
plt.xlim(-0.2, 0.2) 
plt.ylim(-13, 13)

plt.grid(True, linestyle=':', alpha=0.6)

# Legend at center right
plt.legend(loc='center right')

# SAVE THE PLOT RESULT
png_filename = os.path.join(CFG.FIG_DIR, "PLOT_GABUNGAN_Y.png")
plt.savefig(png_filename, dpi=300, bbox_inches='tight')
print(f"\n✅ Square plot saved successfully: {png_filename}")
plt.close()

In [ ]:
# X AXIS ANALYSIS 

import os
import matplotlib.pyplot as plt
import pandas as pd

# 1. PUT 2 DIFFERENT CSV FILE PATHS & NAMES HERE
csv_file1 = CFG.CSV_NASKAH["baseline_x"]
csv_file2 = CFG.CSV_NASKAH["calibrated_x"]

files_to_process = [
    {"path": csv_file1, "label": "Baseline"},
    {"path": csv_file2, "label": "Calibration Result"},
]

print("🔄 STARTING COMBINED X AXIS PLOT PROCESS (1:1 SQUARE)...")

# ✅ CHANGE HERE: Canvas is made square (10x10) for a 1:1 ratio
plt.figure(figsize=(10, 10))

# Horizontal Ground Truth line at Y = 0
plt.axhline(y=0, color='black', linewidth=2, linestyle='--', label='Ground Truth')

colors = ['blue', 'red']

for i, item in enumerate(files_to_process):
    csv_filename = item["path"]
    label_name = item["label"]
    
    if not os.path.exists(csv_filename):
        print(f"❌ File not found: {os.path.basename(csv_filename)}")
        continue
        
    df = pd.read_csv(csv_filename)
    
    # Take the original orientation mapping data
    x_traj = df["X Posisi (mm)"]
    y_traj = df["Y Posisi (mm)"]
    
    # PLOT TRAJECTORY IN ORIGINAL ORIENTATION
    plt.plot(
        x_traj, y_traj, 
        color=colors[i % len(colors)], 
        linewidth=1.0, label=label_name, zorder=2
    )
    
    warna_plot = colors[i % len(colors)]
    
    # =========================================================================
    # 1. LEFT-MOST EXTREME END POINT (MINIMUM X)
    # =========================================================================
    idx_min_x = df["X Posisi (mm)"].idxmin()
    x_kiri = df.loc[idx_min_x, "X Posisi (mm)"]
    y_kiri = df.loc[idx_min_x, "Y Posisi (mm)"]
    
    plt.scatter(x_kiri, y_kiri, color=warna_plot, s=50, marker='o', edgecolor='black', zorder=5)
    
    # Baseline (Blue) directed to Upper Right, Calibrated (Red) to Lower Right
    if i == 0:
        offset_kiri = (15, 20)
        va_align = 'bottom'
    else:
        offset_kiri = (15, -20)
        va_align = 'top'
        
    plt.annotate(
        f"({x_kiri:.2f}, {y_kiri:.2f})", 
        (x_kiri, y_kiri),
        textcoords="offset points", 
        xytext=offset_kiri, 
        ha='left', va=va_align,
        fontsize=12, color=warna_plot, fontweight='bold',
        bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.8, ec="none"),
        arrowprops=dict(arrowstyle="-", color=warna_plot, lw=0.5)
    )
        
    # =========================================================================
    # 2. RIGHT-MOST EXTREME END POINT (MAXIMUM X)
    # =========================================================================
    idx_max_x = df["X Posisi (mm)"].idxmax()
    x_kanan = df.loc[idx_max_x, "X Posisi (mm)"]
    y_kanan = df.loc[idx_max_x, "Y Posisi (mm)"]
    
    plt.scatter(x_kanan, y_kanan, color=warna_plot, s=50, marker='o', edgecolor='black', zorder=5)
    
    # Baseline (Blue) directed to Upper Left, Calibrated (Red) to Lower Left
    if i == 0:
        offset_kanan = (-15, 20)
        va_align_kanan = 'bottom'
    else:
        offset_kanan = (-15, -20)
        va_align_kanan = 'top'
        
    plt.annotate(
        f"({x_kanan:.2f}, {y_kanan:.2f})", 
        (x_kanan, y_kanan),
        textcoords="offset points", 
        xytext=offset_kanan, 
        ha='right', va=va_align_kanan,
        fontsize=12, color=warna_plot, fontweight='bold',
        bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.8, ec="none"),
        arrowprops=dict(arrowstyle="-", color=warna_plot, lw=0.5)
    )

plt.title('2D Trajectory X Axis', fontweight='bold')
plt.xlabel('X Posisi (mm)')
plt.ylabel('Y Posisi (mm)')

# The horizontal axis (X) is given padding so annotations at the left/right ends are not cut off
plt.xlim(-12, 12)
# The vertical axis (Y) is limited to make the error deviation clearer for straight-line motion
plt.ylim(-0.2, 0.2)

plt.grid(True, linestyle=':', alpha=0.6)

# ✅ CHANGE HERE: Move the legend to the upper right so it does not collide with the data line in the lower middle
plt.legend(loc='upper right')

# SAVE THE PLOT RESULT
png_filename = os.path.join(CFG.FIG_DIR, "PLOT_GABUNGAN_X.png")
plt.savefig(png_filename, dpi=300, bbox_inches='tight')
print(f"\n✅ Square original-orientation plot saved successfully: {png_filename}")
plt.close()

In [ ]:
import os
import matplotlib.pyplot as plt
import pandas as pd

# =========================================================================
# 1. PUT 4 CSV PATHS FOR THE X AND Y AXIS GROUPS
# =========================================================================
csv_base_y = CFG.CSV_NASKAH["baseline_y"]
csv_base_x = CFG.CSV_NASKAH["baseline_x"]
csv_kalib_y = CFG.CSV_NASKAH["calibrated_y"]
csv_kalib_x = CFG.CSV_NASKAH["calibrated_x"]

# File parameter configuration
files_to_process = [
    {"path": csv_base_y,  "label": "Baseline Y",       "color": "blue",        "axis": "Y", "pos": "base"},
    {"path": csv_base_x,  "label": "Baseline X",       "color": "deepskyblue", "axis": "X", "pos": "base"},
    {"path": csv_kalib_y, "label": "Calibration Result Y", "color": "red",         "axis": "Y", "pos": "kalib"},
    {"path": csv_kalib_x, "label": "Calibration Result X", "color": "darkred",     "axis": "X", "pos": "kalib"},
]

print("🔄 STARTING COMBINED 2D WORKSPACE PLOT PROCESS (ORIGINAL CSV PRECISION)...")

plt.figure(figsize=(11, 11))

# Ground Truth lines form a cross (+) at the center (0,0)
plt.axhline(y=0, color='black', linewidth=1.5, linestyle='--', label='Ground Truth')
plt.axvline(x=0, color='black', linewidth=1.5, linestyle='--')

for item in files_to_process:
    csv_filename = item["path"]
    label_name   = item["label"]
    warna_plot   = item["color"]
    sumbu        = item["axis"]
    tipe_pos     = item["pos"]
    
    if not os.path.exists(csv_filename):
        print(f"❌ File not found: {os.path.basename(csv_filename)}")
        continue
        
    df = pd.read_csv(csv_filename)
    
    x_traj = df["X Posisi (mm)"]
    y_traj = df["Y Posisi (mm)"]
    
    # PLOT 2D SPATIAL TRAJECTORY
    plt.plot(
        x_traj, y_traj, 
        color=warna_plot, 
        linewidth=2, label=label_name, alpha=0.85, zorder=2
    )
    
    # =========================================================================
    # EXTREME POINT SEARCH & ANNOTATION LOGIC (NO ROUNDING)
    # =========================================================================
    if sumbu == "Y":
        idx_min = y_traj.idxmin()
        idx_max = y_traj.idxmax()
        
        # Because the digit strings are long, the offset shift (x_off) is widened so it does not hit the line
        x_off = -45 if tipe_pos == "base" else 45
        align_ha = 'right' if tipe_pos == "base" else 'left'
        
        # Bottom Extreme
        x_val, y_val = x_traj[idx_min], y_traj[idx_min]
        plt.scatter(x_val, y_val, color=warna_plot, s=45, ec='black', zorder=5)
        plt.annotate(
            f"({x_val}, {y_val})", # Removed the :.2f format to display full precision
            (x_val, y_val),
            textcoords="offset points", xytext=(x_off, -12),
            ha=align_ha, va='top', fontsize=8, color=warna_plot, fontweight='bold',
            bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.85, ec="none"),
            arrowprops=dict(arrowstyle="-", color=warna_plot, lw=0.8)
        )
        
        # Top Extreme
        x_val, y_val = x_traj[idx_max], y_traj[idx_max]
        plt.scatter(x_val, y_val, color=warna_plot, s=45, ec='black', zorder=5)
        plt.annotate(
            f"({x_val}, {y_val})",
            (x_val, y_val),
            textcoords="offset points", xytext=(x_off, 12),
            ha=align_ha, va='bottom', fontsize=8, color=warna_plot, fontweight='bold',
            bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.85, ec="none"),
            arrowprops=dict(arrowstyle="-", color=warna_plot, lw=0.8)
        )

    elif sumbu == "X":
        idx_min = x_traj.idxmin()
        idx_max = x_traj.idxmax()
        
        y_off = 25 if tipe_pos == "base" else -25
        align_va = 'bottom' if tipe_pos == "base" else 'top'
        
        # Left Extreme
        x_val, y_val = x_traj[idx_min], y_traj[idx_min]
        plt.scatter(x_val, y_val, color=warna_plot, s=45, ec='black', zorder=5)
        plt.annotate(
            f"({x_val}, {y_val})",
            (x_val, y_val),
            textcoords="offset points", xytext=(15, y_off),
            ha='left', va=align_va, fontsize=8, color=warna_plot, fontweight='bold',
            bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.85, ec="none"),
            arrowprops=dict(arrowstyle="-", color=warna_plot, lw=0.8)
        )
        
        # Right Extreme
        x_val, y_val = x_traj[idx_max], y_traj[idx_max]
        plt.scatter(x_val, y_val, color=warna_plot, s=45, ec='black', zorder=5)
        plt.annotate(
            f"({x_val}, {y_val})",
            (x_val, y_val),
            textcoords="offset points", xytext=(-15, y_off),
            ha='right', va=align_va, fontsize=8, color=warna_plot, fontweight='bold',
            bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.85, ec="none"),
            arrowprops=dict(arrowstyle="-", color=warna_plot, lw=0.8)
        )

plt.title('2D Workspace Trajectory Evaluation (Full Precision)', fontweight='bold', fontsize=13)
plt.xlabel('X Posisi (mm)')
plt.ylabel('Y Posisi (mm)')

# Axis limits slightly widened because the long number strings need extra horizontal space
plt.xlim(-13.5, 13.5)
plt.ylim(-13.5, 13.5)

plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc='upper left', fontsize='small')

png_filename = os.path.join(CFG.FIG_DIR, "PLOT_GABUNGAN_XY.png")
plt.savefig(png_filename, dpi=300, bbox_inches='tight')
print(f"\n✅ Combined plot with original precision saved successfully: {png_filename}")
plt.close()